# AEON Credit Service Malaysia (ACSM) — Serverless Seamless Data Ingestion into BigQuery
**Run directly inside the BigQuery Studio UI (Colab Enterprise Notebook)**

This notebook demonstrates **100% Serverless, Zero-Service-Provisioning Data Ingestion** into BigQuery in **Singapore (`asia-southeast1`)**:
1. **Step 1**: Set parameterized `PROJECT_ID` (`asia-southeast1` Singapore) & clone the repository.
2. **Step 2**: Create the regional Cloud Storage landing bucket (`gs://acsm-workshop-landing-${PROJECT_ID}`) in **Singapore (`asia-southeast1`)**.
3. **Step 3**: Copy the 8 compressed `.csv.gz` dataset files (`69 MB` / `1,398,284` rows) to the Singapore bucket.
4. **Step 4**: Execute the `CREATE OR REPLACE TABLE` DDL SQL (`00_create_8_tables_ddl_with_descriptions.sql`) to create all **8 tables** (`T1_Fact_EP_Judge` .. `T8_dimProduct`) along with their table and column descriptions.
5. **Step 5**: Execute the Serverless `LOAD DATA OVERWRITE` SQL (`01_load_data_from_gcs.sql`) at **$0 Compute Cost (`0 Bytes Billed`)**.

---
## Step 1: Configure Parameters (`PROJECT_ID` & Singapore Region) and Clone Repository

In [ ]:
# @title Step 1: Set Parameters & Clone Workshop Repository { display-mode: "form" }
import os
import subprocess

PROJECT_ID = "trustedtesterarvind"  # @param {type:"string"}
LOCATION = "asia-southeast1"        # @param ["asia-southeast1"]
DATASET_ID = "acsm_bronze"          # @param {type:"string"}
BUCKET_NAME = f"acsm-workshop-landing-{PROJECT_ID}"

os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["LOCATION"] = LOCATION
os.environ["DATASET_ID"] = DATASET_ID
os.environ["BUCKET_NAME"] = BUCKET_NAME

if not os.path.exists("aeon-credit-gcp-workshop"):
    !git clone -b feature/acsm-e2e-workshop https://github.com/cloud-gtm/aeon-credit-gcp-workshop.git
else:
    !git -C aeon-credit-gcp-workshop pull origin feature/acsm-e2e-workshop

!gcloud config set project {PROJECT_ID}
print(f"\n✅ Configured PROJECT_ID={PROJECT_ID} | LOCATION={LOCATION} (Singapore) | BUCKET=gs://{BUCKET_NAME}")

> **🔍 How to Verify Step 1 on GCP Console UI**
> 1. In the top Google Cloud Console bar, confirm your **`PROJECT_ID`** is selected in the Project Picker.
> 2. In the cell output above, confirm the repository cloned cleanly and `LOCATION=asia-southeast1` (Singapore) is active.
---
## Step 2: Create the Cloud Storage Landing Bucket in Singapore (`asia-southeast1`)

In [ ]:
# @title Step 2: Create Regional GCS Bucket in Singapore (asia-southeast1)
!gcloud storage buckets describe gs://{BUCKET_NAME} --project={PROJECT_ID} >/dev/null 2>&1 || \
  gcloud storage buckets create gs://{BUCKET_NAME} \
    --project={PROJECT_ID} \
    --location={LOCATION} \
    --uniform-bucket-level-access

!gcloud storage buckets describe gs://{BUCKET_NAME} --format="table(name,location,location_type,storage_class)"

> **🔍 How to Verify Step 2 on GCP Console UI (Cloud Storage Browser)**
> 1. Open **Cloud Storage $\rightarrow$ Buckets** in the GCP Console.
> 2. Verify bucket **`acsm-workshop-landing-${PROJECT_ID}`** shows **Location type**: `Region`, **Location**: `asia-southeast1 (Singapore)`, and **Public access**: `Not public`.
---
## Step 3: Copy Compressed Dataset Files (`.csv.gz`) from Repo to Singapore Bucket

In [ ]:
# @title Step 3: Upload All 8 Compressed .csv.gz Files (69 MB / 1.4M Rows) to GCS
!gcloud storage cp aeon-credit-gcp-workshop/data/full_compressed/*.csv.gz gs://{BUCKET_NAME}/full_compressed/
!gcloud storage ls -l gs://{BUCKET_NAME}/full_compressed/

> **🔍 How to Verify Step 3 on GCP Console UI (Bucket Objects View)**
> 1. In **Cloud Storage $\rightarrow$ Buckets**, click **`acsm-workshop-landing-${PROJECT_ID}` $\rightarrow$ `full_compressed/`**.
> 2. Click **Refresh** and verify all `.csv.gz` files (`T1_Fact_EP_Judge.csv.gz` .. `T8_dimProduct.csv.gz`) are listed in `asia-southeast1 (Singapore)` with `application/gzip` content type.
---
## Step 4: Run `CREATE TABLE` DDL Statement (With All Table & 226 Column Descriptions)

In [ ]:
# @title Step 4: Execute 00_create_8_tables_ddl_with_descriptions.sql in Singapore (asia-southeast1)
import pathlib
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
ddl_path = pathlib.Path("aeon-credit-gcp-workshop/track1_platform_governance/00_create_8_tables_ddl_with_descriptions.sql")
ddl_sql = ddl_path.read_text().replace("trustedtesterarvind", PROJECT_ID)

job = client.query(ddl_sql, location=LOCATION)
job.result()
print(f"✅ Step 4 Complete: Created {PROJECT_ID}.{DATASET_ID} and all 8 tables with 226 column descriptions in {LOCATION}.")

> **🔍 How to Verify Step 4 on GCP Console UI (BigQuery Studio Explorer & Schema Tab)**
> 1. In the left **BigQuery Studio Explorer** pane (right beside this notebook!), expand **`${PROJECT_ID}` $\rightarrow$ `acsm_bronze`**.
> 2. Click on **`T1_Fact_EP_Judge`** $\rightarrow$ select the **Schema** tab to verify all **60 columns** have their business **Description** populated from `Mock Metadata.xlsx`, and check the **Details** tab to confirm **Data location** is `asia-southeast1` and **Number of rows** is currently `0`.
---
## Step 5: Run Serverless `LOAD DATA OVERWRITE` Statement (**$0 Load Cost / `0 B Billed`**)
> **💡 Zero Compute Cost (`$0.00` / `0 Bytes Billed`)**: Batch loading data into BigQuery from Cloud Storage via the `LOAD DATA` SQL statement is **100% FREE (`$0.00`)** using BigQuery's shared batch slot pool ([BigQuery Pricing](https://cloud.google.com/bigquery/pricing#loading_data)). Because both the bucket and dataset are in **Singapore (`asia-southeast1`)**, there is **$0 network egress cost** and **100% of the 226 column descriptions** from Step 4 are preserved automatically.

In [ ]:
# @title Step 5: Execute 01_load_data_from_gcs.sql (Serverless LOAD DATA OVERWRITE — 0 Bytes Billed)
load_path = pathlib.Path("aeon-credit-gcp-workshop/track1_platform_governance/01_load_data_from_gcs.sql")
load_sql = load_path.read_text().replace("trustedtesterarvind", PROJECT_ID)

load_job = client.query(load_sql, location=LOCATION)
load_job.result()
bytes_billed = load_job.total_bytes_billed or 0
print(f"✅ Step 5 Complete! Loaded all 8 tables in {LOCATION} | Total Bytes Billed: {bytes_billed} B ($0.00 FREE Serverless Batch Load)")

In [ ]:
# @title Step 5 Verification: Audit Table Row Counts & 226 Column Descriptions via INFORMATION_SCHEMA
audit_sql = f"""
SELECT
  t.table_name,
  s.row_count,
  COUNT(c.column_name) AS total_columns,
  COUNTIF(c.description IS NOT NULL AND c.description != '') AS described_columns,
  REGEXP_REPLACE(COALESCE(opt.option_value, ''), r'^"|"$', '') AS table_description
FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.TABLES` t
JOIN `{PROJECT_ID}.{DATASET_ID}.__TABLES__` s
  ON t.table_name = s.table_id
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.TABLE_OPTIONS` opt
  ON t.table_name = opt.table_name AND opt.option_name = 'description'
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS` c
  ON t.table_name = c.table_name
WHERE STARTS_WITH(t.table_name, 'T')
GROUP BY 1, 2, 5
ORDER BY 1
"""
df_audit = client.query(audit_sql, location=LOCATION).to_dataframe()
print(f"Total Rows Loaded Across 8 Tables: {df_audit['row_count'].sum():,d} | Total Described Columns: {df_audit['described_columns'].sum()}/{df_audit['total_columns'].sum()} (100.0%)")
display(df_audit)

> **🔍 How to Verify Step 5 on GCP Console UI (4 Visual Checks in BigQuery Studio)**
> 1. **Verify `$0` Load Cost (`0 B Billed`)**: In the cell output above (or in BigQuery **Job history**), point out **`Total Bytes Billed: 0 B ($0.00 FREE Serverless Batch Load)`** in **`asia-southeast1`**.
> 2. **Verify Row Counts (`Details` Tab)**: In the left Explorer tree, click **`T5_Fact_CC_Sales`** ($535,925$ rows) or **`T1_Fact_EP_Judge`** ($140,000$ rows) $\rightarrow$ **Details** tab.
> 3. **Verify Loaded Records (`Preview` Tab — also $0 Cost)**: Click the **Preview** tab on any table to browse the records at zero query cost (`0 B billed`).
> 4. **Verify Preserved Column Descriptions (`Schema` Tab)**: Click the **Schema** tab to confirm all **226 column descriptions** remained intact after `LOAD DATA OVERWRITE`.